# Harmonic Functions and the Laplace Equation

A function $u: \Omega \to \mathbb{R}$ is **harmonic** if it satisfies the **Laplace equation**:
$$
\Delta u = \frac{\partial^2 u}{\partial x^2} + \frac{\partial^2 u}{\partial y^2} = 0 \quad \text{in } \Omega.
$$

## Mean value property

Harmonic functions satisfy the **mean value property**: the value at any point equals the average over any sphere centred at that point:
$$
u(x_0) = \frac{1}{|\partial B_r|} \int_{\partial B_r(x_0)} u \, d\sigma.
$$
This immediately implies the **maximum principle**: a harmonic function attains its maximum (and minimum) on the boundary $\partial\Omega$.

## Discrete harmonic functions

On a grid, the discrete Laplacian is $(\Delta_h u)_{i,j} = u_{i+1,j} + u_{i-1,j} + u_{i,j+1} + u_{i,j-1} - 4u_{i,j}$. The discrete harmonic condition $\Delta_h u = 0$ says each interior value is the average of its four neighbours — a **four-point stencil mean value property**.

## Boundary value problem

Given boundary values $u|_{\partial\Omega} = g$, the **Dirichlet problem** has a unique harmonic solution. Numerically it reduces to a sparse linear system $(D-A)u_{\text{int}} = -A_{\text{bd}} g$ where $A$ is the adjacency matrix of the grid graph.

## Conformal maps

In 2D, harmonic functions are the real (or imaginary) parts of **holomorphic functions**. Level sets of $u$ are orthogonal to level sets of the harmonic conjugate $v$ (where $u + iv$ is holomorphic).

## Environment

In [ ]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

plt.rcParams['figure.dpi'] = 120

## Solving the Dirichlet problem on a square

We set boundary conditions on a square $[0,1]^2$ grid and solve the discrete Laplace equation.

In [ ]:
def solve_laplace(n, bc_fn):
    """
    Solve Delta u = 0 on [0,1]^2 grid with Dirichlet BC.
    bc_fn(i, j, n): returns boundary value at grid point (i,j).
    """
    N = n * n
    # Index map: interior nodes only
    is_boundary = np.zeros((n, n), dtype=bool)
    is_boundary[0, :] = True; is_boundary[-1, :] = True
    is_boundary[:, 0] = True; is_boundary[:, -1] = True
    interior = ~is_boundary

    # Build BC values
    u = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if is_boundary[i, j]:
                u[i, j] = bc_fn(i, j, n)

    # Build sparse Laplacian for interior nodes
    idx = np.full((n, n), -1)
    idx[interior] = np.arange(interior.sum())
    n_int = interior.sum()

    rows, cols_sp, vals = [], [], []
    rhs = np.zeros(n_int)

    for i in range(1, n-1):
        for j in range(1, n-1):
            r = idx[i, j]
            rows.append(r); cols_sp.append(r); vals.append(4.0)
            for di, dj in [(-1,0),(1,0),(0,-1),(0,1)]:
                ni2, nj2 = i+di, j+dj
                if interior[ni2, nj2]:
                    rows.append(r); cols_sp.append(idx[ni2,nj2]); vals.append(-1.0)
                else:
                    rhs[r] += u[ni2, nj2]

    A = sp.csr_matrix((vals, (rows, cols_sp)), shape=(n_int, n_int))
    u_int = spla.spsolve(A, rhs)
    u[interior] = u_int
    return u

# Test: linear BC
n = 60
u_lin = solve_laplace(n, lambda i, j, n: j / (n-1))
print('Max deviation from linear (should be ~0):', np.abs(u_lin - np.tile(np.linspace(0,1,n), (n,1))).max())

## Gallery of boundary conditions

We solve the Laplace equation for several interesting boundary conditions.

In [ ]:
def bc_top_hot(i, j, n):
    """Top edge = 1, rest = 0."""
    return 1.0 if i == 0 else 0.0

def bc_fourier(i, j, n, k=3):
    """Sinusoidal BC on left and right edges."""
    x, y = j/(n-1), i/(n-1)
    if i == 0:   return np.sin(k * np.pi * x)
    if i == n-1: return -np.sin(k * np.pi * x)
    return 0.0

def bc_checkerboard(i, j, n):
    """Alternating ±1 on boundary."""
    x, y = j/(n-1), i/(n-1)
    if i == 0 or i == n-1:
        return np.sin(4 * np.pi * x)
    return np.sin(4 * np.pi * y) if j == 0 or j == n-1 else 0.0

def bc_corner(i, j, n):
    """Top-left quadrant = 1, rest = 0."""
    return 1.0 if (i == 0 and j < n//2) or (j == 0 and i < n//2) else 0.0

bcs = [bc_top_hot, bc_fourier, bc_checkerboard, bc_corner]
titles = ['Top edge hot', 'Fourier BC', 'Oscillating BC', 'Corner sources']

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for col, (bc, title) in enumerate(zip(bcs, titles)):
    u = solve_laplace(n, bc)
    axes[0, col].imshow(u, cmap='RdBu_r', origin='upper', vmin=-1, vmax=1)
    axes[0, col].axis('off'); axes[0, col].set_title(title, fontsize=9)
    axes[1, col].contourf(u[::-1], levels=16, cmap='RdBu_r')
    axes[1, col].contour(u[::-1], levels=16, colors='k', linewidths=0.4, alpha=0.5)
    axes[1, col].axis('off')

fig.suptitle('Harmonic functions: solutions to $\\Delta u = 0$', y=1.02)
plt.tight_layout(); plt.show()

## Harmonic conjugate and conformal mapping

For a harmonic $u$, the conjugate $v$ (satisfying Cauchy-Riemann equations) can be found by solving $\nabla v = (-u_y, u_x)$. The level sets of $u$ and $v$ are orthogonal.

In [ ]:
# Use the Fourier BC solution
u_c = solve_laplace(n, bc_fourier)

# Compute gradient and rotate to get harmonic conjugate gradient
uy, ux = np.gradient(u_c)
vx, vy = -uy, ux   # Cauchy-Riemann: v_x = -u_y, v_y = u_x

# Integrate: v = integral of (vx dx + vy dy) — approximate via cumsum
v = np.cumsum(vy, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
for ax, (func, title, cmap) in zip(axes, [(u_c, '$u$ (harmonic)', 'RdBu_r'),
                                           (v,  '$v$ (conjugate)', 'PuOr')]):
    x_g = np.linspace(0, 1, n)
    ax.contourf(x_g, x_g, func[::-1], levels=14, cmap=cmap)
    ax.contour(x_g, x_g, u_c[::-1], levels=10, colors='k',   lw=0.7, alpha=0.5)
    ax.contour(x_g, x_g, v[::-1],   levels=10, colors='red', linewidths=0.7, alpha=0.5)
    ax.set_aspect('equal'); ax.axis('off'); ax.set_title(title)

fig.suptitle('Harmonic $u$ (black level sets) and conjugate $v$ (red level sets) — orthogonal families')
plt.tight_layout(); plt.show()

## Interactive: sinusoidal boundary frequency

In [ ]:
def show_harmonic(k=3, n_grid=50):
    u_i = solve_laplace(n_grid, lambda i,j,n: bc_fourier(i,j,n,k=k))
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    axes[0].imshow(u_i, cmap='RdBu_r', origin='upper', vmin=-1, vmax=1)
    axes[0].axis('off'); axes[0].set_title(f'Harmonic $u$ (BC freq $k={k}$)')
    x_g = np.linspace(0, 1, n_grid)
    axes[1].contourf(x_g, x_g[::-1], u_i, levels=16, cmap='RdBu_r')
    axes[1].contour(x_g, x_g[::-1], u_i, levels=16, colors='k', linewidths=0.4, alpha=0.6)
    axes[1].set_aspect('equal'); axes[1].axis('off')
    axes[1].set_title('Level sets')
    plt.tight_layout(); plt.show()

interact(show_harmonic,
         k=IntSlider(value=3, min=1, max=8, step=1, description='freq $k$'),
         n_grid=IntSlider(value=50, min=20, max=80, step=10, description='grid $n$'));

## Bibliographical resources

- Axler, S., Bourdon, P. and Ramey, W. (2001). *Harmonic Function Theory* (2nd ed.). Springer.
- Evans, L. C. (2010). *Partial Differential Equations* (2nd ed.). American Mathematical Society.
- Needham, T. (1997). *Visual Complex Analysis*. Oxford University Press.
- Stein, E. M. and Weiss, G. (1971). *Introduction to Fourier Analysis on Euclidean Spaces*. Princeton University Press.
- LeVeque, R. J. (2007). *Finite Difference Methods for Ordinary and Partial Differential Equations*. SIAM.